# State-space models — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def make_series(n=60, trend=0.08, amp=2.0, noise=0.25):
    t=np.arange(n)
    return t, 10 + trend*t + amp*np.sin(2*np.pi*t/12) + noise*np.random.randn(n)
def acf(x, max_lag):
    x=np.asarray(x)-np.mean(x); den=np.dot(x,x)
    return np.array([1.0]+[np.dot(x[:-k],x[k:])/den for k in range(1,max_lag+1)])
def moving_average(x,w):
    return np.convolve(x, np.ones(w)/w, mode='valid')
print('setup ok')

## A hidden state evolves forward, noisy observations correct it, and uncertainty decides how much to trust each side
State-space forecasting separates a latent process from observations. These examples show prediction, Kalman gain, correction, uncertainty shrinkage, and tracking a drifting local level.

In [ ]:
# 1) prediction step: uncertainty grows by process noise Q
m=10; P=4; Q=1; pred_m=m; pred_P=P+Q
plt.figure(figsize=(6,3)); plt.bar(['old P','predicted P'],[P,pred_P]); plt.title('process noise widens uncertainty')
assert pred_P==5

In [ ]:
# 2) Kalman gain balances predicted uncertainty against measurement noise
P_pred=5; R=3; K=P_pred/(P_pred+R)
plt.figure(figsize=(6,3)); plt.bar(['P_pred','R','K'],[P_pred,R,K]); plt.title('gain = 5/(5+3)=0.625')
assert abs(K-0.625)<1e-12

In [ ]:
# 3) update step pulls the state toward the observation
m_pred=10; y=13; K=0.625; m_upd=m_pred+K*(y-m_pred)
plt.figure(figsize=(6,3)); plt.bar(['pred','obs','updated'],[m_pred,y,m_upd]); plt.title('innovation is only partly trusted')
assert abs(m_upd-11.875)<1e-12

In [ ]:
# 4) posterior uncertainty shrinks after observing
P_pred=5; K=0.625; P_upd=(1-K)*P_pred
plt.figure(figsize=(6,3)); plt.bar(['before obs','after obs'],[P_pred,P_upd]); plt.title('observations reduce uncertainty')
assert abs(P_upd-1.875)<1e-12

In [ ]:
# 5) local-level filter tracks a noisy drifting series
true=10+0.15*np.arange(60); obs=true+0.6*np.random.randn(60); m=obs[0]; P=1.; Q=0.05; R=0.36; filt=[]
for z in obs:
    P=P+Q; K=P/(P+R); m=m+K*(z-m); P=(1-K)*P; filt.append(m)
plt.figure(figsize=(6,3)); plt.plot(obs,alpha=.4,label='observed'); plt.plot(true,label='true'); plt.plot(filt,label='filtered'); plt.legend(); plt.title('state-space smoothing by uncertainty')
assert np.mean((np.array(filt)-true)**2) < np.mean((obs-true)**2)